# aiohappyeyeballs: 3 Basic Usage Examples

In [ ]:
# Example 1: Connect with Happy Eyeballs to a local listener (no network needed)
import asyncio
import socket
import aiohappyeyeballs

async def greet(reader, writer):
    writer.write(b"hello")
    await writer.drain()
    writer.close()

server = await asyncio.start_server(greet, "127.0.0.1", 0)  # port 0 = any free port
port = server.sockets[0].getsockname()[1]
loop = asyncio.get_running_loop()
try:
    addr_infos = await loop.getaddrinfo("127.0.0.1", port, type=socket.SOCK_STREAM)
    sock = await aiohappyeyeballs.start_connection(addr_infos, happy_eyeballs_delay=0.25)
    try:
        assert sock.getpeername() == ("127.0.0.1", port), sock.getpeername()
        greeting = await loop.sock_recv(sock, 5)
        assert greeting == b"hello", greeting
        print(f"Connected socket: {sock.getsockname()} -> {sock.getpeername()}, received {greeting!r}")
    finally:
        sock.close()
finally:
    server.close()
    await server.wait_closed()


In [ ]:
# Example 2: Manipulate address info lists
import aiohappyeyeballs

# Two candidate addresses for one host: an IPv4 one and an IPv6 one.
addr_infos = aiohappyeyeballs.addr_to_addr_infos(("127.0.0.1", 8080)) + aiohappyeyeballs.addr_to_addr_infos(("::1", 8080))
print(f"Original address infos: {len(addr_infos)} entries")
assert [info[4] for info in addr_infos] == [("127.0.0.1", 8080), ("::1", 8080, 0, 0)], addr_infos

# remove_addr_infos drops the address that failed (typically sock.getpeername()) and edits the list in place.
aiohappyeyeballs.remove_addr_infos(addr_infos, ("127.0.0.1", 8080))
assert [info[4] for info in addr_infos] == [("::1", 8080, 0, 0)], addr_infos
print(f"After remove_addr_infos: {len(addr_infos)} entry left: {addr_infos[0][4]}")

try:
    aiohappyeyeballs.remove_addr_infos(addr_infos, ("10.0.0.1", 8080))
    raise AssertionError("removing an address that is not in the list must raise")
except ValueError as error:
    print(f"Removing an unknown address raises ValueError: {error}")


In [ ]:
# Example 3: Convert a local address to address info
import aiohappyeyeballs

local_addr = ('127.0.0.1', 0)
local_addr_infos = aiohappyeyeballs.addr_to_addr_infos(local_addr)
print(f'Local addr_infos: {local_addr_infos}')
